# 05 · Approach 3: Pretrained self-supervised transformer (wav2vec 2.0 / XLS-R) on raw audio
Owner: **Member 4** · Runtime: Colab **GPU** (T4 is enough; XLS-R-300M takes the longest)

**Why this model.** wav2vec 2.0 (Baevski et al., 2020) learns speech representations from raw
waveforms: a convolutional encoder turns audio into ~50 frames/s, and a **transformer** relates every
frame to every other frame with self-attention, so dependencies across the whole utterance are modelled
directly rather than passed step by step as in an RNN. It is pretrained without labels on thousands of
hours, which matters in a low-resource setting like ours. XLS-R (Babu et al., 2022) is pretrained on
128 languages, including Swahili sources such as Common Voice and BABEL, so we compare English-only vs
multilingual pretraining. The winning Zindi hackathon solution also used a wav2vec 2.0 encoder with a
classification head (Zindi, 2023).
**Limitations:** ~95M–300M parameters (vs <1M for the CNN/RNN), slower training and inference, and
pretraining data that is mostly read speech, not phone recordings.

## 0 · Setup
Run this cell first. On **Google Colab** it clones the repo, installs requirements and copies the
data from your Google Drive to the (much faster) local Colab disk. Locally it does nothing special.

**Before the first run:** put `Train.csv`, `Test.csv`, `SampleSubmission.csv` and `Audio.zip`
(downloaded from Zindi) in a Google Drive folder called `swahili_audio`, and set `REPO_URL`.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/YOUR-GROUP/swahili-audio-sequential.git"   # <-- edit once
    REPO_DIR = Path("/content/swahili-audio-sequential")
    DRIVE_DATA = Path("/content/drive/MyDrive/swahili_audio")
    LOCAL_DATA = Path("/content/data")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if not (LOCAL_DATA / ".ready").exists():
        LOCAL_DATA.mkdir(parents=True, exist_ok=True)
        for f in DRIVE_DATA.glob("*.csv"):
            shutil.copy(f, LOCAL_DATA)
        subprocess.run(["unzip", "-q", "-o", str(DRIVE_DATA / "Audio.zip"), "-d", str(LOCAL_DATA)], check=True)
        (LOCAL_DATA / ".ready").touch()
    os.environ["DATA_DIR"] = str(LOCAL_DATA)
    os.environ["CACHE_DIR"] = str(DRIVE_DATA / "cache")          # features persist between sessions
    os.environ["OUTPUT_BACKUP_DIR"] = str(DRIVE_DATA / "outputs")  # results backed up to Drive
else:
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import os
import numpy as np
import torch

from src import config as C
from src.data import get_splits, label_encoder, encode, load_unlabelled, add_paths
from src.datasets import ArrayDataset, Normalize, WaveAugment
from src.features import compute_features
from src.models import HFAudioClassifier
from src.pipeline import run_experiments, finalize
from src.utils import set_seed, get_device

MEMBER, MODEL = "Member 4", "wav2vec2 transformer"
TINY = os.environ.get("TINY_W2V") == "1"      # offline smoke test only (random tiny model)
set_seed(C.SEED)
print("Device:", get_device())
df = get_splits()
classes, to_id = label_encoder(df.label)
K = len(classes)
tr, va, te = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
ytr, yva, yte = (encode(d.label, to_id) for d in (tr, va, te))
zindi = add_paths(load_unlabelled())

## Input: raw 16 kHz waveform, silence-trimmed, fixed length, per-clip normalised

In [ ]:
Wtr, Wva, Wte, Wz = (compute_features(d.path, "wave") for d in (tr, va, te, zindi))
print("waveform array:", Wtr.shape, f"({Wtr.nbytes / 1e9:.2f} GB)")


def make_datasets(p):
    aug = Normalize(WaveAugment()) if p.get("augment") else Normalize()
    return ArrayDataset(Wtr, ytr, aug), ArrayDataset(Wva, yva, Normalize())


def build_model(p):
    return HFAudioClassifier(p["checkpoint"], K, freeze=p.get("freeze", "feature_encoder"), tiny_random=TINY)

## Experiments
Batch size × gradient accumulation = effective batch 32 for all fine-tuning runs.

In [ ]:
BASE, XLSR = "facebook/wav2vec2-base", "facebook/wav2vec2-xls-r-300m"
DEFAULT_TRAIN = dict(epochs=15, lr=5e-5, batch_size=16, grad_accum=2, weight_decay=1e-2, patience=4,
                     num_workers=2)
EXPERIMENTS = [
    dict(id="W2V-01", description="wav2vec2-base (English pretraining), frozen, train head only",
         hypothesis="Self-supervised features are already word-discriminative without fine-tuning.",
         motivated_by="Starting point: how good are the pretrained representations as-is?",
         params={"checkpoint": BASE, "freeze": "base"}, train={"lr": 1e-3, "epochs": 20}),
    dict(id="W2V-02", description="wav2vec2-base fine-tuned (CNN feature encoder frozen)",
         hypothesis="Adapting the transformer layers to Swahili words and phone audio closes the gap.",
         motivated_by="W2V-01: frozen English features leave errors on Swahili-specific sounds",
         params={"checkpoint": BASE}),
    dict(id="W2V-03", description="XLS-R-300M (multilingual pretraining incl. Swahili) fine-tuned",
         hypothesis="Pretraining that includes Swahili and many accents transfers better than English-only.",
         motivated_by="W2V-02 vs W2V-01 showed fine-tuning helps; now test the pretraining language",
         params={"checkpoint": XLSR}, train={"batch_size": 8, "grad_accum": 4, "lr": 3e-5}),
    dict(id="W2V-04", description="Best checkpoint + waveform augmentation (gain, shift, noise)",
         hypothesis="Noise/gain augmentation improves robustness to low-SNR phone recordings (EDA section 5).",
         motivated_by="EDA: a noticeable share of recordings are noisy or quiet",
         params={"checkpoint": XLSR, "augment": True}, train={"batch_size": 8, "grad_accum": 4, "lr": 3e-5}),
]
# TIP: if W2V-02 beats W2V-03, change W2V-04 to use BASE (the "best checkpoint") and re-run it.
if TINY:
    for e in EXPERIMENTS:
        e.setdefault("train", {}).update(batch_size=8, grad_accum=1)
best, summary = run_experiments(EXPERIMENTS, build_model, make_datasets, member=MEMBER, model_name=MODEL,
                                n_classes=K, y_train=ytr, default_train=DEFAULT_TRAIN)
summary.round(4)

> Fill the `observation` column in `experiments/log_wav2vec2_transformer.csv`. Key questions:
> frozen vs fine-tuned; English vs multilingual pretraining; did noise augmentation help, and on
> which clips? Also note training time and parameter count: this is the cost side of the trade-off.

## Final evaluation on the test split (once)

In [ ]:
model, proba_test, metrics = finalize(best, build_model, ArrayDataset(Wte, yte, Normalize()), yte,
                                      te.file.values, classes, key="wav2vec2_transformer",
                                      display_name="wav2vec 2.0 / XLS-R",
                                      zindi_ds=ArrayDataset(Wz, None, Normalize()), zindi_df=zindi)

In [ ]:
from src.utils import persist_outputs
persist_outputs()